# Capstone G4: Skin cancer AI and who gets access to it

Same 7-class skin-lesion model as the screening group -- but your question is about **equity**: who does a dermatology AI actually help, and who gets left out?

**Your group's priority: fairness and access.** Dermatology AI is often trained mostly on light skin and deployed where dermatologists already exist. A model can be accurate on average and still widen the gap for the people who most lack access to a specialist.

This notebook gives you a **working baseline**. Your job is to make it better and be able to
explain every change. Use Claude as your pair programmer. **Rubric:** build it, evaluate it
honestly, find a failure mode, defend one design decision, both partners can explain the work.

In [ ]:
# Setup: on Colab this grabs the course files + installs what's missing. Locally it's a no-op.
import os, sys
if not os.path.exists("../../capstone_common.py"):
    os.system("git clone -q https://github.com/jinchiwei/outset-ai-healthcare.git")
    os.chdir("outset-ai-healthcare/notebooks/day3_capstone/projects/g4_skin_access")
sys.path.insert(0, "../..")            # day3_capstone (capstone_common / _tabular / _seq)
sys.path.insert(0, "../../../_shared") # colab_setup
import colab_setup
colab_setup.ensure(*colab_setup.DAY1, "medmnist")

In [ ]:
import torch, sys
sys.path.insert(0, "../..")
import capstone_common as cc

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

FLAG = "dermamnist"
train_loader, val_loader, test_loader, n_classes, task = cc.get_loaders(FLAG, size=64)
CLASS_NAMES = cc.class_names(FLAG)
print("classes:", n_classes, "->", CLASS_NAMES, "| task:", task)

## A look at the data
Before modeling, always look. What do these images actually show?

In [ ]:
import matplotlib.pyplot as plt
imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for ax, im, lb in zip(axes, imgs, labels):
    ax.imshow(im[0], cmap="gray")
    ax.set_title(CLASS_NAMES[int(lb)], fontsize=8); ax.axis("off")
plt.tight_layout()

## Baseline: transfer learning (frozen ResNet18 + new head)
The Day 1 trick. Run it, note the TEST accuracy, then try to beat it.

In [ ]:
model = cc.make_baseline(n_classes)
model = cc.train(model, train_loader, val_loader, epochs=3, lr=1e-3, device=device)
test_acc = cc.evaluate(model, test_loader, device=device)["accuracy"]
print(f"\nbaseline TEST accuracy: {test_acc:.3f}")

## Build your own model (interactive)

**Pick options, hit Run, read the TEST accuracy.** Change **one** thing at a time and watch
what actually moves the number. Log every result -- that log is half your presentation.

- **backbone** -- the architecture (resnet18 is small/fast; resnet50, densenet are bigger).
- **pretrained** -- start from ImageNet weights (on) or scratch (off). *Usually matters a lot.*
- **unfreeze** -- train the whole network vs just a new head. More power, more overfitting.
- **augment** -- add flips/rotations. **epochs** -- how long to train.

In [ ]:
from ipywidgets import interact_manual, Dropdown, Checkbox, IntSlider

def build_model(backbone="resnet18", pretrained=True, unfreeze_backbone=False, augment=False, epochs=3):
    global model, train_loader, val_loader, test_loader
    train_loader, val_loader, test_loader, n_cls, _ = cc.get_loaders(FLAG, size=64, augment=augment)
    model = cc.make_model(n_cls, backbone=backbone, pretrained=pretrained, unfreeze_backbone=unfreeze_backbone)
    model = cc.train(model, train_loader, val_loader,
                     epochs=epochs, lr=(1e-4 if unfreeze_backbone else 1e-3), device=device)
    acc = cc.evaluate(model, test_loader, device=device)["accuracy"]
    print(f"\n>>> TEST accuracy = {acc:.3f}   "
          f"[{backbone}, pretrained={pretrained}, unfreeze={unfreeze_backbone}, augment={augment}, {epochs}ep]")

try:
    interact_manual(build_model,
        backbone=Dropdown(options=cc.BACKBONES, value="resnet18", description="backbone"),
        pretrained=Checkbox(value=True, description="pretrained"),
        unfreeze_backbone=Checkbox(value=False, description="unfreeze"),
        augment=Checkbox(value=False, description="augment"),
        epochs=IntSlider(value=3, min=1, max=10, description="epochs"))
except ImportError:
    build_model()

## The regulator's toolkit: audit your own model

Yesterday you set the *rules* medical AI must follow. Now hold **your** model to them. Your
group's priority is **fairness and access**, so the dropdown starts on **Fairness (accuracy per class)** --
but run the others too. A high accuracy is not enough if it fails one of these.

- **Fairness** -- is accuracy even across all 7 lesion types? Uneven accuracy across conditions is an access problem: the model helps some patients more than others.
- **Failure analysis** -- who does it fail, and would they be able to get a second opinion?
- Then confusion matrix / Grad-CAM / monitoring.

*(Train a model first -- run the baseline or the builder above.)*

In [ ]:
from ipywidgets import interact_manual, Dropdown

TOOLS = list(cc.REGULATOR_TOOLS)
DEFAULT = [t for t in TOOLS if t.startswith("Fairness")][0]

def audit(tool):
    if "model" not in globals():
        print("Train a model first (run the baseline or the builder above)."); return
    cc.REGULATOR_TOOLS[tool](model, test_loader, device=device, class_names=CLASS_NAMES)

try:
    interact_manual(audit, tool=Dropdown(options=TOOLS, value=DEFAULT,
                                          description="priority", style={"description_width": "initial"}))
except ImportError:
    audit(DEFAULT)

## Where to take it (with Claude)
- Research the **skin-tone gap** in dermatology datasets (Fitzpatrick scale). Does MedMNIST even tell you the skin tones it was trained on? (It doesn't -- like G3, that gap is a finding.)
- Argue it out: if this model runs in a clinic with no dermatologist, does it help or harm? Use your fairness numbers to defend a position.